# 自动化重训练教程

> **前置知识**: Python基础、机器学习训练流程、数据漂移概念
>
> **学习目标**: 掌握模型自动重训练的触发策略和实现方法

---

## 为什么需要自动重训练？

```
模型效果下降的常见原因:
┌─────────────────────────────────────────────────────────────┐
│  数据漂移 (Data Drift)                                      │
│  └── 线上数据分布与训练数据不同                            │
│                                                             │
│  概念漂移 (Concept Drift)                                   │
│  └── 特征与标签的关系发生变化                              │
│                                                             │
│  季节性变化                                                  │
│  └── 用户行为随时间周期性变化                              │
│                                                             │
│  业务变化                                                    │
│  └── 新产品、新用户群体、新场景                            │
└─────────────────────────────────────────────────────────────┘

自动重训练解决方案:
┌─────────────────────────────────────────────────────────────┐
│  监控 → 检测 → 触发 → 重训练 → 验证 → 部署                │
│                                                             │
│  触发策略:                                                  │
│  ├── 定时触发: 每周/每月固定重训练                         │
│  ├── 漂移触发: 检测到数据漂移时触发                        │
│  └── 性能触发: 模型指标下降超过阈值时触发                  │
└─────────────────────────────────────────────────────────────┘
```

## 本教程内容

1. **重训练触发策略** - 定时、漂移、性能三种触发方式
2. **增量学习** - 在新数据上持续更新模型
3. **持续学习** - 防止灾难性遗忘的经验回放
4. **完整重训练流水线** - 端到端自动化流程

In [ ]:
# ============================================================
# 环境准备
# ============================================================
# 标准库
import numpy as np
import time
from enum import Enum
from dataclasses import dataclass
from typing import Callable, List, Dict

# 机器学习库
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import accuracy_score

print("=" * 50)
print("环境准备完成")
print("=" * 50)
print(f"NumPy 版本: {np.__version__}")

## 1. 重训练触发策略

**核心概念**: 根据不同条件自动决定何时重新训练模型

```
三种触发策略对比:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  定时触发 (Scheduled)                                       │
│  ├── 原理: 固定时间间隔重训练（如每周、每月）              │
│  ├── 优点: 简单可靠，易于规划资源                          │
│  ├── 缺点: 可能浪费资源或响应不及时                        │
│  └── 适用: 数据稳定增长的场景                              │
│                                                             │
│  漂移触发 (Drift-based)                                     │
│  ├── 原理: 检测到数据漂移时触发                            │
│  ├── 优点: 及时响应数据变化                                │
│  ├── 缺点: 需要漂移检测机制                                │
│  └── 适用: 数据分布可能突变的场景                          │
│                                                             │
│  性能触发 (Performance-based)                               │
│  ├── 原理: 模型指标下降超过阈值时触发                      │
│  ├── 优点: 直接关联业务效果                                │
│  ├── 缺点: 需要实时标签反馈，有延迟                        │
│  └── 适用: 有快速标签反馈的场景                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 重训练触发策略实现
# ============================================================

class RetrainTrigger(Enum):
    """
    重训练触发类型枚举
    
    三种触发策略:
    - SCHEDULED: 定时触发，固定时间间隔
    - DRIFT: 漂移触发，检测到数据漂移时
    - PERFORMANCE: 性能触发，模型效果下降时
    """
    SCHEDULED = "scheduled"      # 定时触发
    DRIFT = "drift"              # 漂移触发
    PERFORMANCE = "performance"  # 性能触发


@dataclass
class RetrainConfig:
    """
    重训练配置
    
    参数说明:
    ┌─────────────────────────────────────────────────────────┐
    │  trigger: 触发策略类型                                  │
    │  schedule_interval: 定时触发间隔（秒），默认1小时       │
    │  drift_threshold: 漂移阈值（PSI），默认0.2             │
    │  performance_threshold: 性能下降阈值，默认0.05 (5%)    │
    │  min_samples: 最小样本数，确保统计可靠性               │
    └─────────────────────────────────────────────────────────┘
    """
    trigger: RetrainTrigger
    schedule_interval: int = 3600      # 定时间隔（秒）
    drift_threshold: float = 0.2       # 漂移阈值
    performance_threshold: float = 0.05 # 性能下降阈值
    min_samples: int = 100             # 最小样本数


class RetrainManager:
    """
    重训练管理器
    
    核心功能:
    1. should_retrain(): 判断是否需要重训练
    2. record_retrain(): 记录重训练完成，更新基线
    
    决策流程:
    ┌─────────────────────────────────────────────────────────┐
    │  1. 检查样本量是否足够                                 │
    │  2. 根据触发策略检查条件:                              │
    │     - SCHEDULED: 检查时间间隔                          │
    │     - DRIFT: 检查漂移分数                              │
    │     - PERFORMANCE: 检查性能下降                        │
    │  3. 返回决策和原因                                     │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(self, config: RetrainConfig):
        self.config = config
        self.last_retrain = time.time()  # 上次重训练时间
        self.baseline_perf = None        # 基线性能
        self.samples_count = 0           # 累计样本数
    
    def should_retrain(self, drift_score: float = 0, current_perf: float = 0) -> tuple:
        """
        判断是否需要重训练
        
        参数:
            drift_score: 漂移分数（如 PSI）
            current_perf: 当前模型性能
            
        返回:
            (should_retrain, reason): 是否重训练及原因
        """
        # 检查样本量
        if self.samples_count < self.config.min_samples:
            return False, f"样本量不足 ({self.samples_count}/{self.config.min_samples})"
        
        # 定时触发
        if self.config.trigger == RetrainTrigger.SCHEDULED:
            elapsed = time.time() - self.last_retrain
            if elapsed > self.config.schedule_interval:
                return True, f"定时触发 (已过 {elapsed/3600:.1f} 小时)"
        
        # 漂移触发
        elif self.config.trigger == RetrainTrigger.DRIFT:
            if drift_score > self.config.drift_threshold:
                return True, f"漂移触发 (PSI={drift_score:.3f} > {self.config.drift_threshold})"
        
        # 性能触发
        elif self.config.trigger == RetrainTrigger.PERFORMANCE:
            if self.baseline_perf is not None:
                drop = self.baseline_perf - current_perf
                if drop > self.config.performance_threshold:
                    return True, f"性能触发 (下降 {drop:.1%} > {self.config.performance_threshold:.1%})"
        
        return False, "无需重训练"
    
    def record_retrain(self, new_perf: float):
        """
        记录重训练完成
        
        参数:
            new_perf: 新模型的性能，作为新的基线
        """
        self.last_retrain = time.time()
        self.baseline_perf = new_perf
        self.samples_count = 0
        print(f"重训练完成，新基线性能: {new_perf:.3f}")


# ============================================================
# 重训练触发演示
# ============================================================
print("=" * 60)
print("重训练触发策略演示")
print("=" * 60)

# 场景: 性能触发
config = RetrainConfig(
    trigger=RetrainTrigger.PERFORMANCE,
    performance_threshold=0.05  # 性能下降超过 5% 触发
)
manager = RetrainManager(config)
manager.baseline_perf = 0.95  # 基线性能 95%
manager.samples_count = 200   # 已收集 200 个样本

print("\n性能触发测试:")
print("-" * 40)

# 测试1: 性能下降 3%（不触发）
result, reason = manager.should_retrain(current_perf=0.92)
print(f"当前性能 92% (下降 3%): {result}, {reason}")

# 测试2: 性能下降 7%（触发）
result, reason = manager.should_retrain(current_perf=0.88)
print(f"当前性能 88% (下降 7%): {result}, {reason}")

## 2. 增量学习

**核心概念**: 增量学习（Incremental Learning）是在新数据到达时持续更新模型，而不是从头重新训练

```
增量学习 vs 全量重训练:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  全量重训练:                                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  旧数据 + 新数据 → 从头训练 → 新模型                │   │
│  │  优点: 模型质量稳定                                 │   │
│  │  缺点: 计算成本高，训练时间长                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  增量学习:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  旧模型 + 新数据 → 继续训练 → 更新后的模型          │   │
│  │  优点: 计算成本低，更新快速                         │   │
│  │  缺点: 可能遗忘旧知识（灾难性遗忘）                 │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘

增量学习适用场景:
- 流式数据（数据持续到达）
- 计算资源有限
- 需要快速响应数据变化
- 数据量太大无法全量训练
```

In [ ]:
# ============================================================
# 增量学习器实现
# ============================================================

class IncrementalLearner:
    """
    增量学习器
    
    核心思想: 使用支持增量更新的模型（如 SGDClassifier），
    在新数据到达时继续训练，而不是从头开始
    
    工作流程:
    ┌─────────────────────────────────────────────────────────┐
    │  新数据到达 → 加入缓冲区 → 缓冲区满? → 增量训练       │
    │                              │                          │
    │                              ↓                          │
    │                         保留部分旧数据（防止遗忘）      │
    └─────────────────────────────────────────────────────────┘
    
    关键参数:
    - buffer_size: 缓冲区大小，达到此大小时触发训练
    - 保留比例: 训练后保留 20% 的数据，防止完全遗忘
    """
    
    def __init__(self, buffer_size: int = 500):
        """
        参数:
            buffer_size: 缓冲区大小，累积到此数量后触发训练
        """
        # 使用 SGDClassifier，支持 partial_fit 增量训练
        self.model = SGDClassifier(loss='log_loss', random_state=42)
        self.buffer_size = buffer_size
        self.X_buffer = []  # 特征缓冲区
        self.y_buffer = []  # 标签缓冲区
        self.is_fitted = False  # 模型是否已初始化
    
    def partial_fit(self, X, y, classes=None) -> bool:
        """
        增量训练
        
        参数:
            X: 新数据特征
            y: 新数据标签
            classes: 所有可能的类别（首次训练时必须提供）
            
        返回:
            是否触发了训练
        """
        # 将新数据加入缓冲区
        self.X_buffer.extend(X)
        self.y_buffer.extend(y)
        
        # 检查是否达到训练阈值
        if len(self.X_buffer) >= self.buffer_size:
            X_train = np.array(self.X_buffer)
            y_train = np.array(self.y_buffer)
            
            # 增量训练
            if not self.is_fitted:
                # 首次训练，需要指定所有类别
                self.model.partial_fit(X_train, y_train, 
                                       classes=classes or np.unique(y_train))
                self.is_fitted = True
            else:
                # 后续训练，继续更新模型
                self.model.partial_fit(X_train, y_train)
            
            # 保留部分数据防止遗忘（保留 20%）
            keep = self.buffer_size // 5
            self.X_buffer = self.X_buffer[-keep:]
            self.y_buffer = self.y_buffer[-keep:]
            
            return True  # 触发了训练
        
        return False  # 未触发训练
    
    def predict(self, X):
        """预测"""
        if not self.is_fitted:
            raise ValueError("模型尚未训练")
        return self.model.predict(X)


# ============================================================
# 增量学习演示
# ============================================================
print("=" * 60)
print("增量学习演示")
print("=" * 60)

# 创建增量学习器
learner = IncrementalLearner(buffer_size=200)

print("\n模拟流式数据训练:")
print("-" * 40)

# 模拟 10 批数据流
for batch in range(10):
    # 生成一批数据（每批 50 个样本）
    X = np.random.randn(50, 10)
    y = (X[:, 0] > 0).astype(int)  # 简单的分类规则
    
    # 增量训练
    trained = learner.partial_fit(X, y, classes=[0, 1])
    
    if trained:
        print(f"  批次 {batch}: 触发训练 (缓冲区已满)")
    else:
        print(f"  批次 {batch}: 数据已缓存 (缓冲区: {len(learner.X_buffer)})")

# 测试模型效果
print("\n" + "-" * 40)
print("模型测试:")
X_test = np.random.randn(100, 10)
y_test = (X_test[:, 0] > 0).astype(int)
y_pred = learner.predict(X_test)
print(f"  测试准确率: {accuracy_score(y_test, y_pred):.3f}")

## 3. 持续学习（防止灾难性遗忘）

**核心问题**: 灾难性遗忘（Catastrophic Forgetting）是指模型在学习新任务时，会忘记之前学过的知识

```
灾难性遗忘问题:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  任务1训练 → 模型学会任务1 (准确率 95%)                    │
│       ↓                                                     │
│  任务2训练 → 模型学会任务2 (准确率 90%)                    │
│       ↓                                                     │
│  测试任务1 → 准确率下降到 60%! (遗忘了任务1)              │
│                                                             │
└─────────────────────────────────────────────────────────────┘

经验回放 (Experience Replay) 解决方案:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  核心思想: 保留一部分旧数据，训练时混合新旧数据            │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  记忆库 (Memory Buffer)                             │   │
│  │  ├── 存储历史数据样本                               │   │
│  │  ├── 容量有限，需要采样策略                         │   │
│  │  └── 训练时混合新数据和记忆数据                     │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  训练流程:                                                  │
│  新数据 + 记忆数据 → 混合训练 → 更新模型 → 更新记忆库    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 持续学习器实现（经验回放）
# ============================================================

class ContinualLearner:
    """
    持续学习器 - 使用经验回放防止灾难性遗忘
    
    核心思想: 维护一个记忆库，训练时混合新数据和历史数据
    
    工作流程:
    ┌─────────────────────────────────────────────────────────┐
    │  新数据到达                                             │
    │       ↓                                                 │
    │  合并新数据 + 记忆库数据                                │
    │       ↓                                                 │
    │  混合训练                                               │
    │       ↓                                                 │
    │  更新记忆库（随机采样保留）                             │
    └─────────────────────────────────────────────────────────┘
    
    记忆库管理策略:
    - 容量有限，超出时随机删除旧样本
    - 保证各类别样本的平衡（可选）
    """
    
    def __init__(self, memory_size: int = 500):
        """
        参数:
            memory_size: 记忆库容量，存储历史样本数量
        """
        self.model = SGDClassifier(loss='log_loss', random_state=42)
        self.memory_size = memory_size
        self.memory_X = []  # 记忆库特征
        self.memory_y = []  # 记忆库标签
        self.is_fitted = False
    
    def learn(self, X_new, y_new):
        """
        学习新数据（同时保留旧知识）
        
        参数:
            X_new: 新数据特征
            y_new: 新数据标签
        """
        # 合并新数据和记忆库数据
        if self.memory_X:
            X_combined = np.vstack([np.array(self.memory_X), X_new])
            y_combined = np.hstack([np.array(self.memory_y), y_new])
        else:
            X_combined = X_new
            y_combined = y_new
        
        # 训练模型
        if not self.is_fitted:
            self.model.partial_fit(X_combined, y_combined, 
                                   classes=np.unique(y_combined))
            self.is_fitted = True
        else:
            self.model.partial_fit(X_combined, y_combined)
        
        # 更新记忆库
        self._update_memory(X_new, y_new)
    
    def _update_memory(self, X, y):
        """
        更新记忆库
        
        策略: 添加新样本，超出容量时随机删除
        """
        # 添加新样本到记忆库
        for i in range(len(X)):
            self.memory_X.append(X[i])
            self.memory_y.append(y[i])
        
        # 超出容量时随机删除
        while len(self.memory_X) > self.memory_size:
            idx = np.random.randint(len(self.memory_X))
            self.memory_X.pop(idx)
            self.memory_y.pop(idx)
    
    def predict(self, X):
        """预测"""
        return self.model.predict(X)


# ============================================================
# 持续学习演示：多任务学习
# ============================================================
print("=" * 60)
print("持续学习演示（防止灾难性遗忘）")
print("=" * 60)

# 创建持续学习器
continual = ContinualLearner(memory_size=300)

# ============================================================
# 任务1: 学习正态分布数据
# ============================================================
print("\n任务1: 学习正态分布数据")
print("-" * 40)
X1 = np.random.randn(200, 10)
y1 = (X1[:, 0] > 0).astype(int)  # 规则: 第一个特征 > 0
continual.learn(X1, y1)
print(f"  训练样本: 200")
print(f"  分类规则: X[:, 0] > 0")

# ============================================================
# 任务2: 学习不同分布的数据
# ============================================================
print("\n任务2: 学习偏移分布数据")
print("-" * 40)
X2 = np.random.randn(200, 10) + 2  # 分布偏移
y2 = (X2[:, 1] > 2).astype(int)   # 规则: 第二个特征 > 2
continual.learn(X2, y2)
print(f"  训练样本: 200")
print(f"  分类规则: X[:, 1] > 2")

# ============================================================
# 测试两个任务的效果
# ============================================================
print("\n" + "=" * 60)
print("测试结果（验证是否遗忘）")
print("=" * 60)

# 测试任务1
X1_test = np.random.randn(100, 10)
y1_test = (X1_test[:, 0] > 0).astype(int)
acc1 = accuracy_score(y1_test, continual.predict(X1_test))
print(f"\n任务1 准确率: {acc1:.3f}")
print(f"  (如果没有经验回放，这个值会很低)")

# 测试任务2
X2_test = np.random.randn(100, 10) + 2
y2_test = (X2_test[:, 1] > 2).astype(int)
acc2 = accuracy_score(y2_test, continual.predict(X2_test))
print(f"\n任务2 准确率: {acc2:.3f}")

print(f"\n记忆库大小: {len(continual.memory_X)} 样本")

## 4. 完整重训练流水线

**核心概念**: 将触发策略、训练、评估、部署整合成端到端的自动化流程

```
自动重训练流水线架构:
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  ┌─────────┐    ┌─────────┐    ┌─────────┐    ┌─────────┐  │
│  │ 数据流  │ →  │ 触发器  │ →  │ 训练器  │ →  │ 评估器  │  │
│  └─────────┘    └─────────┘    └─────────┘    └─────────┘  │
│       │              │              │              │        │
│       ▼              ▼              ▼              ▼        │
│   收集数据      检查条件       重新训练       验证效果     │
│                                                    │        │
│                                              ┌─────┴─────┐  │
│                                              ▼           ▼  │
│                                           通过?       失败? │
│                                              │           │  │
│                                              ▼           ▼  │
│                                           部署新模型  保留旧模型│
│                                                             │
└─────────────────────────────────────────────────────────────┘

流水线组件:
┌─────────────────────────────────────────────────────────────┐
│  model_factory: 模型工厂函数，创建新模型实例               │
│  eval_fn: 评估函数，计算模型性能                           │
│  retrain_config: 重训练配置（触发策略、阈值等）            │
│  history: 历史记录，追踪所有版本的性能                     │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 自动重训练流水线实现
# ============================================================

class AutoRetrainPipeline:
    """
    自动重训练流水线
    
    核心功能:
    1. initial_train(): 初始训练，建立基线
    2. check_and_retrain(): 检查条件并决定是否重训练
    3. _retrain(): 执行重训练
    
    流水线流程:
    ┌─────────────────────────────────────────────────────────┐
    │  初始训练 → 建立基线                                   │
    │       ↓                                                 │
    │  新数据到达 → 评估当前模型                             │
    │       ↓                                                 │
    │  检查触发条件:                                          │
    │  ├── 性能下降超过阈值? → 重训练                        │
    │  ├── 检测到数据漂移? → 重训练                          │
    │  └── 都不满足 → 继续监控                               │
    │       ↓                                                 │
    │  重训练 → 更新模型版本 → 记录历史                      │
    └─────────────────────────────────────────────────────────┘
    """
    
    def __init__(
        self,
        model_factory: Callable,
        eval_fn: Callable,
        retrain_config: RetrainConfig
    ):
        """
        参数:
            model_factory: 模型工厂函数，返回新的模型实例
            eval_fn: 评估函数，签名 eval_fn(model, X, y) -> float
            retrain_config: 重训练配置
        """
        self.model_factory = model_factory
        self.eval_fn = eval_fn
        self.config = retrain_config
        self.current_model = None    # 当前模型
        self.model_version = 0       # 模型版本号
        self.history = []            # 历史记录
    
    def initial_train(self, X, y) -> float:
        """
        初始训练
        
        参数:
            X: 训练特征
            y: 训练标签
            
        返回:
            初始模型的性能
        """
        # 创建并训练模型
        self.current_model = self.model_factory()
        self.current_model.fit(X, y)
        
        # 评估性能
        perf = self.eval_fn(self.current_model, X, y)
        
        # 记录版本
        self.model_version = 1
        self.history.append({
            'version': 1,
            'performance': perf,
            'time': time.time(),
            'reason': '初始训练'
        })
        
        print(f"初始模型 v{self.model_version}，性能: {perf:.3f}")
        return perf
    
    def check_and_retrain(self, X_new, y_new, drift_score: float = 0) -> float:
        """
        检查条件并决定是否重训练
        
        参数:
            X_new: 新数据特征
            y_new: 新数据标签
            drift_score: 漂移分数（可选）
            
        返回:
            当前/新模型的性能
        """
        # 评估当前模型在新数据上的性能
        current_perf = self.eval_fn(self.current_model, X_new, y_new)
        baseline = self.history[-1]['performance'] if self.history else 0
        
        should_retrain = False
        reason = ""
        
        # 检查性能下降
        perf_drop = baseline - current_perf
        if perf_drop > self.config.performance_threshold:
            should_retrain = True
            reason = f"性能下降 {perf_drop:.1%}"
        
        # 检查漂移
        if drift_score > self.config.drift_threshold:
            should_retrain = True
            reason = f"数据漂移 (PSI={drift_score:.3f})"
        
        # 执行重训练
        if should_retrain:
            print(f"\n触发重训练: {reason}")
            return self._retrain(X_new, y_new, reason)
        
        print(f"当前性能: {current_perf:.3f}，无需重训练")
        return current_perf
    
    def _retrain(self, X, y, reason: str) -> float:
        """
        执行重训练
        
        参数:
            X: 训练特征
            y: 训练标签
            reason: 重训练原因
            
        返回:
            新模型的性能
        """
        # 创建并训练新模型
        new_model = self.model_factory()
        new_model.fit(X, y)
        
        # 评估新模型
        new_perf = self.eval_fn(new_model, X, y)
        
        # 更新当前模型
        self.current_model = new_model
        self.model_version += 1
        
        # 记录历史
        self.history.append({
            'version': self.model_version,
            'performance': new_perf,
            'time': time.time(),
            'reason': reason
        })
        
        print(f"重训练完成: v{self.model_version}，性能: {new_perf:.3f}")
        return new_perf


# ============================================================
# 自动重训练流水线演示
# ============================================================
print("=" * 60)
print("自动重训练流水线演示")
print("=" * 60)

# 配置
config = RetrainConfig(
    trigger=RetrainTrigger.PERFORMANCE,
    performance_threshold=0.1  # 性能下降超过 10% 触发重训练
)

# 创建流水线
pipeline = AutoRetrainPipeline(
    model_factory=lambda: LogisticRegression(max_iter=1000),
    eval_fn=lambda m, X, y: accuracy_score(y, m.predict(X)),
    retrain_config=config
)

# ============================================================
# 阶段1: 初始训练
# ============================================================
print("\n阶段1: 初始训练")
print("-" * 40)
X_init = np.random.randn(500, 10)
y_init = (X_init[:, 0] > 0).astype(int)
pipeline.initial_train(X_init, y_init)

# ============================================================
# 阶段2: 模拟正常数据（不触发重训练）
# ============================================================
print("\n阶段2: 正常数据（相同分布）")
print("-" * 40)
X_normal = np.random.randn(500, 10)
y_normal = (X_normal[:, 0] > 0).astype(int)
pipeline.check_and_retrain(X_normal, y_normal)

# ============================================================
# 阶段3: 模拟数据漂移（触发重训练）
# ============================================================
print("\n阶段3: 数据漂移（分布偏移）")
print("-" * 40)
X_drift = np.random.randn(500, 10) + 1  # 分布偏移
y_drift = (X_drift[:, 0] > 1).astype(int)  # 决策边界也变了
pipeline.check_and_retrain(X_drift, y_drift)

# ============================================================
# 查看历史记录
# ============================================================
print("\n" + "=" * 60)
print("模型版本历史")
print("=" * 60)
for record in pipeline.history:
    print(f"  v{record['version']}: 性能={record['performance']:.3f}, 原因={record['reason']}")

## 总结

本教程介绍了自动重训练的核心概念和实现方法：

### 核心组件回顾

| 组件 | 功能 | 关键方法 |
|:-----|:-----|:---------|
| RetrainTrigger | 触发策略枚举 | SCHEDULED, DRIFT, PERFORMANCE |
| RetrainConfig | 重训练配置 | 阈值、间隔、最小样本数 |
| RetrainManager | 触发判断 | should_retrain() |
| IncrementalLearner | 增量学习 | partial_fit() |
| ContinualLearner | 持续学习 | learn(), 经验回放 |
| AutoRetrainPipeline | 完整流水线 | check_and_retrain() |

### 重训练策略对比

| 策略 | 适用场景 | 优点 | 缺点 |
|:-----|:---------|:-----|:-----|
| 定时重训练 | 稳定环境 | 简单可靠 | 可能浪费资源 |
| 漂移触发 | 动态环境 | 及时响应 | 需要漂移检测 |
| 性能触发 | 有标签反馈 | 精准高效 | 反馈延迟 |
| 增量学习 | 流式数据 | 计算高效 | 可能遗忘 |
| 持续学习 | 多任务场景 | 保留知识 | 复杂度高 |

### 最佳实践

```
自动重训练检查清单:
✓ 设置合理的触发阈值（性能下降 5-10%）
✓ 确保最小样本量（100-500）
✓ 记录每次重训练的原因和效果
✓ 保留历史模型版本用于回滚
✓ 在重训练前验证新数据质量
✓ 重训练后进行 A/B 测试验证

常见陷阱:
✗ 阈值设置过低导致频繁重训练
✗ 忽略灾难性遗忘问题
✗ 没有验证新模型效果就直接部署
✗ 没有保留旧模型用于回滚
```

### 下一步学习

- **07_FeatureStore_tutorial.ipynb**: 特征存储与数据版本控制